In [2]:

from getpass import getpass
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.storage import InMemoryStore
from langchain_core.documents import Document

GOOGLE_API_KEY = getpass("Enter Gemini API Key: ")

docs = [
    Document(
        page_content="LangChain helps build LLM-powered apps with memory and agents.",
        metadata={"id": "1"}
    ),
    Document(
        page_content="Agents in LangChain use tools to answer questions.",
        metadata={"id": "2"}
    )
]

# 3. Setup child splitter
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

# 4. Setup Gemini embeddings
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

# 5. Setup vectorstore
vectorstore = FAISS.from_texts(["placeholder"], embedding)
vectorstore.delete([vectorstore.index_to_docstore_id[0]])

# 6. Parent retriever
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=InMemoryStore(),
    child_splitter=child_splitter
)

# 7. Add documents
retriever.add_documents(docs)

results = retriever.invoke("What are agents?")

for doc in results:
    print("Retrieved Doc:", doc.page_content)


/tmp/ipykernel_3151/503055988.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Enter Gemini API Key: ··········
Retrieved Doc: Agents in LangChain use tools to answer questions.
Retrieved Doc: LangChain helps build LLM-powered apps with memory and agents.


In [1]:
%pip install -q --no-deps langchain-google-genai==4.4.3
%pip install -q langchain-community==0.4.2 faiss-cpu
%pip install -q filetype

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 4.4.3 requires google-genai>=2.20.0, but you have google-genai 2.12.1 which is incompatible.
langchain-google-genai 4.4.3 requires langchain-core<2.0.0,>=1.7.5, but you have langchain-core 1.7.4 which is incompatible.


In [6]:

from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# Create simple text docs
docs = [
    Document(page_content="LangChain enables LLM applications."),
    Document(page_content="Vector search is powerful."),
    Document(page_content="BM25 is a classical retrieval method.")
]

bm25_retriever = BM25Retriever.from_documents(docs)

# Retrieve
results = bm25_retriever.invoke("How does BM25 work?")

for doc in results:
    print("BM25 Result:", doc.page_content)


BM25 Result: BM25 is a classical retrieval method.
BM25 Result: Vector search is powerful.
BM25 Result: LangChain enables LLM applications.


In [5]:

%pip install -U rank-bm25


In [7]:

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

# Sample docs
docs = [
    Document(page_content="LangChain supports LLMs."),
    Document(page_content="You can build AI apps using LangChain.")
]

# BM25 Retriever
bm25 = BM25Retriever.from_documents(docs)

# Vector Retriever
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

vectorstore = FAISS.from_documents(docs, embedding)
vector_retriever = vectorstore.as_retriever()

# Ensemble Retriever (equal weight)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

# Retrieve
results = ensemble_retriever.invoke("AI apps using LangChain")

for doc in results:
    print("Ensemble Doc:", doc.page_content)


Ensemble Doc: You can build AI apps using LangChain.
Ensemble Doc: LangChain supports LLMs.


In [8]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

docs = [
    Document(
        page_content="LangChain is for LLM-based apps",
        metadata={"last_accessed_at": datetime.now()}
    ),
    Document(
        page_content="Vector search improves relevance",
        metadata={"last_accessed_at": datetime.now()}
    )
]

vectorstore = FAISS.from_documents(docs, embedding)
retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

results = retriever.invoke("What is LangChain?")

for r in results:
    print(r)
    print(r.page_content)


In [11]:
!pip install tavily-python

In [12]:

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_core.documents import Document
from datetime import datetime

embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

docs = [
    Document(
        page_content="LangChain is for LLM-based apps",
        metadata={"last_accessed_at": datetime.now()}
    ),
    Document(
        page_content="Vector search improves relevance",
        metadata={"last_accessed_at": datetime.now()}
    )
]

vectorstore = FAISS.from_documents(docs, embedding)

retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vectorstore,
    decay_rate=0.01,
    k=2,
    score_threshold=None
)

results = retriever.invoke("What is LangChain?")

for r in results:
    print(r)
    print(r.page_content)


In [13]:

from getpass import getpass
from langchain_community.retrievers import TavilySearchAPIRetriever
import os

TAVILY_API_KEY = getpass("Enter Tavily API Key: ")
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

retriever = TavilySearchAPIRetriever(k=3)

docs = retriever.invoke("Latest updates about LangChain")

for doc in docs:
    print(doc.page_content)


Enter Tavily API Key: ··········
release(langchain): 1.5.1 (#41943)  
fix(core): keep warning suppression context-local (#41881)  
fix(langchain): avoid warning-filter races in provider inference (#41880)  
fix(langchain): avoid warning-filter races in `ModelRequest` (#41879)  
chore(deps): bump opentelemetry-api from 1.45.1 to 1.45.2 in /libs/langchain\_v1 (#41845)  
chore(deps): bump dnspython from 2.9.1 to 2.9.2 in /libs/langchain\_v1 (#41849)

## langchain-xai==1.4.1

## langchain-openai==1.8.2

## langchain-groq==1.2.1 [...] release(anthropic): 1.7.11 (#41929)  
feat(anthropic): send `langsmith_gateway` headers from the client (#41902)  
feat(anthropic): support Claude Fable and Mythos 5.2 (#41770)  
fix(anthropic): emit a chunk when an empty thinking block starts (#41907)  
feat(partners): Remove EOL'd Python 3.10 from partner packages (#41268)  
chore(model-profiles): refresh model profile data (#41791)  
chore(model-profiles): refresh model profile data (#41781)
Use per-call mo